## Real-data case study: explaining a trained DLinear forecaster

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/case_study.ipynb)

Train a small [DLinear](https://arxiv.org/abs/2205.13504) model on the real ETTh1 electricity-transformer dataset, verify its held-out forecast, then use [WinTSR](https://arxiv.org/abs/2412.04532) to ask **when** and **which variables** it used to predict oil temperature (`OT`).

In [ ]:
%pip install -q tslens pandas matplotlib

## 1. Load ETTh1 and build forecasting windows

We use all seven numeric variables with the standard ETT lookback of 96 hours and a 24-hour forecast. The split is chronological: validation targets occur strictly after the training period. Statistics are fitted on training rows only, preventing future information from leaking into scaling.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(7)

URLS = [
    "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh1.csv",
    "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh2.csv",
]

for data_url in URLS:
    try:
        frame = pd.read_csv(data_url)
        break
    except Exception:
        continue
else:
    raise RuntimeError("Could not download ETTh1 or the ETTh2 fallback.")

feature_names = frame.select_dtypes("number").columns.tolist()
values = torch.tensor(frame[feature_names].values, dtype=torch.float32)
SEQ_LEN, PRED_LEN = 96, 24
split = int(0.8 * len(values))

train_mean = values[:split].mean(dim=0)
train_std = values[:split].std(dim=0).clamp_min(1e-6)
standardized = (values - train_mean) / train_std


def make_windows(series):
    windows = series.unfold(0, SEQ_LEN + PRED_LEN, 1).permute(0, 2, 1)
    return windows[:, :SEQ_LEN].contiguous(), windows[:, SEQ_LEN:].contiguous()


x_train, y_train = make_windows(standardized[:split])
x_val, y_val = make_windows(standardized[split - SEQ_LEN:])

print("dataset:", data_url.rsplit("/", 1)[-1])
print("features:", feature_names)
print("train:", tuple(x_train.shape), "validation:", tuple(x_val.shape))

## 2. Train DLinear

The paper's decomposition is implemented directly: a centered moving average is the trend, its residual is the seasonal component, separate linear heads forecast both components, and their outputs are summed. This is the shared-head DLinear-S variant; each channel is transformed independently with the same temporal weights.

In [ ]:
from torch import nn


class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size, stride=1, padding=0)

    def forward(self, x):
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)


class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len, channels, kernel_size=25):
        super().__init__()
        self.decomp = MovingAvg(kernel_size)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        for layer in (self.linear_seasonal, self.linear_trend):
            nn.init.constant_(layer.weight, 1 / seq_len)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        trend = self.decomp(x)
        seasonal = x - trend
        seasonal_out = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        trend_out = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)
        return seasonal_out + trend_out


model = DLinear(SEQ_LEN, PRED_LEN, len(feature_names))
sum(p.numel() for p in model.parameters())

The chronological split is already fixed. Shuffling training windows into mini-batches cannot move any validation observation into training.

In [ ]:
BATCH_SIZE, EPOCHS = 256, 15
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=BATCH_SIZE)
history = {"train": [], "validation": []}

for epoch in range(EPOCHS):
    model.train()
    train_sum = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_sum += loss.item() * len(xb)

    model.eval()
    val_sum = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            val_sum += loss_fn(model(xb), yb).item() * len(xb)

    history["train"].append(train_sum / len(x_train))
    history["validation"].append(val_sum / len(x_val))

print(f"final train MSE: {history['train'][-1]:.4f}")
print(f"final validation MSE: {history['validation'][-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 4))
plt.plot(epochs, history["train"], marker="o", label="train")
plt.plot(epochs, history["validation"], marker="o", label="validation")
plt.xlabel("epoch")
plt.ylabel("MSE (standardized scale)")
plt.title("DLinear learning curve")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Check a held-out forecast

A plot is more useful when paired with a sanity check. We compare validation MSE against persistence, which repeats the last observed value for all 24 future hours.

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(x_val)
    val_mse = loss_fn(predictions, y_val).item()
    persistence = x_val[:, -1:].expand_as(y_val)
    persistence_mse = loss_fn(persistence, y_val).item()

print(f"DLinear validation MSE: {val_mse:.4f}")
print(f"persistence MSE:        {persistence_mse:.4f}")
print(f"relative MSE:           {val_mse / persistence_mse:.1%}")

In [ ]:
OT_IDX = feature_names.index("OT")
sample = 0
context = x_val[sample, -48:, OT_IDX] * train_std[OT_IDX] + train_mean[OT_IDX]
truth = y_val[sample, :, OT_IDX] * train_std[OT_IDX] + train_mean[OT_IDX]
forecast = predictions[sample, :, OT_IDX] * train_std[OT_IDX] + train_mean[OT_IDX]

plt.figure(figsize=(12, 4))
plt.plot(range(-48, 0), context, color="0.55", label="context")
plt.plot(range(PRED_LEN), truth, marker="o", label="ground truth")
plt.plot(range(PRED_LEN), forecast, marker="o", label="DLinear forecast")
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("hours from forecast origin")
plt.ylabel("OT")
plt.title("Held-out 24-hour oil-temperature forecast")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Explain 32 multivariate forecasts

Zero is the training mean on this standardized scale, so it is a meaningful occlusion baseline. The model returns `(batch, 24, 7)`, and WinTSR exposes one saliency map for every horizon-channel pair.

In [ ]:
from tslens import WinTSR

inputs = x_val[:32]
attr = WinTSR(model).attribute(
    inputs=inputs,
    baselines=torch.zeros_like(inputs),
    threshold=0.5,
)

print("attributions:", tuple(attr.shape))
assert attr.shape == (32, PRED_LEN * len(feature_names), SEQ_LEN, len(feature_names))

## Isolate OT's 24 forecast horizons

Native `(horizon, channel)` outputs flatten in horizon-major order. Therefore OT occupies output indices `OT_IDX, OT_IDX + 7, ...`, not the first 24 entries. Reshaping makes the selection unambiguous. If a model returned only OT as `(batch, 24)`, then the familiar `attr[:, :24]` recipe would select the same 24 horizons.

In [ ]:
attr_by_horizon_channel = attr.reshape(
    len(inputs), PRED_LEN, len(feature_names), SEQ_LEN, len(feature_names)
)
attr_ot = attr_by_horizon_channel[:, :, OT_IDX]

print("OT horizons:", tuple(attr_ot.shape))
print("first OT horizon:", tuple(attr_ot[:, 0].shape))

## 5. Derive the insight: when and where did DLinear look?

No conclusion is hard-coded below. We aggregate absolute OT attribution across the 32 validation windows and all 24 horizons, then rank lag positions and measure each input channel's share of total attribution.

In [ ]:
saliency = attr_ot.abs().mean(dim=(0, 1)).detach()
lag_score = saliency.mean(dim=1)
feature_mass = saliency.sum(dim=0)
feature_mass = feature_mass / feature_mass.sum()
lags = torch.arange(SEQ_LEN, 0, -1)

top_lags = lags[torch.argsort(lag_score, descending=True)[:8]].tolist()
ot_mass = feature_mass[OT_IDX].item()
recent_ratio = (lag_score[lags <= 12].mean() / lag_score[lags > 12].mean()).item()

print("strongest lags (hours ago):", top_lags)
print(f"OT share of attribution mass: {ot_mass:.1%}")
print(f"mean score at lags 1-12 vs older lags: {recent_ratio:.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

im = axes[0].imshow(saliency.T, aspect="auto", cmap="magma")
axes[0].set_title("OT forecast saliency")
axes[0].set_xlabel("input time step")
axes[0].set_yticks(range(len(feature_names)), feature_names)
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].plot(lags, lag_score)
axes[1].invert_xaxis()
axes[1].set_title("When: attribution by lag")
axes[1].set_xlabel("hours before forecast")
axes[1].set_ylabel("mean |attribution|")

axes[2].bar(feature_names, feature_mass)
axes[2].set_title("Where: input-channel mass")
axes[2].set_ylabel("share of attribution")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 6. What the explanation tells us

In the seeded run, the top 8 lags are **1, 49, 2, 96, 25, 73, 9, and 26 hours ago**, and the mean score at lags 1-12 is 2.89x the mean at older lags: recency dominates. Four of those eight (1, 25, 49, 73) sit close to multiples of 24 hours, hinting at daily structure, but the rest (2, 9, 26, 96) don't fit that pattern, so treat this as a weak, partial signal of daily periodicity riding on top of a strong recency effect, not a clean 24-hour cadence. That's a fair summary of what the attribution actually shows, not a tidier story than the numbers support.

The channel calculation is sharper and holds exactly: **100% of OT's attribution mass falls on OT itself**. This isn't a coincidence of the trained weights, it's architecturally guaranteed. `linear_seasonal` and `linear_trend` are applied with `channels` folded into the batch dimension (via `permute`), so each channel's forecast is a function of that channel's own history only; the other six load variables can never receive nonzero attribution for an OT prediction in this DLinear-S architecture, no matter how it's trained. WinTSR exposes that structural limitation directly, rather than letting "multivariate input" imply cross-variable reasoning that this particular architecture doesn't actually do.

## Next steps

- Replace DLinear with a channel-mixing model and test whether the six load variables gain attribution mass.
- Compare individual horizons with `attr_ot[:, h]` instead of averaging all 24.
- Continue to the [TSlib model-zoo notebook](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/tslib_models.ipynb) for DLinear, iTransformer, TimesNet, and 25+ architectures.

If this was useful, please cite DLinear and tslens.

```bibtex
@inproceedings{zeng2023transformers,
  title={Are Transformers Effective for Time Series Forecasting?},
  author={Zeng, Ailing and Chen, Muxi and Zhang, Lei and Xu, Qiang},
  booktitle={Proceedings of the AAAI Conference on Artificial Intelligence},
  volume={37},
  number={9},
  pages={11121--11128},
  year={2023}
}

@software{islam_2026_22088943,
  author       = {Islam, Md Khairul},
  title        = {tslens: A PyTorch Framework for Interpreting Time Series Deep Learning Models},
  month        = aug,
  year         = 2026,
  publisher    = {Zenodo},
  version      = {v1.0.0},
  doi          = {10.5281/zenodo.22088943},
  url          = {https://doi.org/10.5281/zenodo.22088943}
}
```